In [3]:
!pip install fastavro

In [4]:
import random
import datetime
import json
from collections import defaultdict

# ============================================================
# Milestone 1: Streaming Data Feed Design & Generation
# ============================================================

random.seed(7)

# -----------------------------
# Config (tune these easily)
# -----------------------------
NUM_ORDER_EVENTS = 4000          # total events in the order lifecycle stream
NUM_COURIER_EVENTS = 4000        # total events in the courier status/location stream

NUM_RESTAURANTS = 120
NUM_COURIERS = 300
NUM_CUSTOMERS = 800

ZONES = ["Z1_Center", "Z2_North", "Z3_South", "Z4_East", "Z5_West"]
ZONE_WEIGHTS = [0.40, 0.18, 0.15, 0.14, 0.13]  # skew: more demand in Center

# Lunch/dinner peak shaping (higher = more likely)
LUNCH_HOURS = list(range(12, 15))   # 12-14
DINNER_HOURS = list(range(19, 23))  # 19-22

WEEKEND_MULTIPLIER = 1.25
SURGE_PROB = 0.08                   # chance of demand surge periods
PROMO_PROB = 0.10                   # chance of promo periods
BASE_CANCEL_PROB = 0.06             # baseline cancellation probability
SURGE_CANCEL_BONUS = 0.05           # extra cancels during surge
PROMO_CANCEL_BONUS = -0.02          # fewer cancels during promo (example)
DUPLICATE_EVENT_PROB = 0.02
LATE_EVENT_PROB = 0.05              # out-of-order arrival (event_time behind ingestion_time)
MISSING_STEP_PROB = 0.03            # skip some lifecycle steps
IMPOSSIBLE_DURATION_PROB = 0.01     # anomalies for detection
COURIER_OFFLINE_MID_DELIVERY_PROB = 0.02

START_DATE = datetime.datetime(2026, 2, 1, 0, 0, 0)
END_DATE   = datetime.datetime(2026, 3, 1, 0, 0, 0)

# ============================================================
# Feed A: Order Lifecycle Events (JSON + AVRO schema)
# ============================================================

order_schema_v1 = {
    "type": "record",
    "name": "OrderLifecycleEvent",
    "namespace": "food.delivery",
    "doc": "Order lifecycle events for real-time food delivery analytics",
    "fields": [
        {"name": "schema_version", "type": "string", "default": "1.0"},
        {"name": "event_id", "type": "string"},
        {"name": "order_id", "type": "string"},
        {"name": "event_type", "type": {"type": "enum", "name": "OrderEventType",
                                        "symbols": ["CREATED", "ACCEPTED", "PREP_STARTED", "READY",
                                                    "PICKED_UP", "DELIVERED", "CANCELLED"]}},
        {"name": "event_time", "type": "string"},      # ISO timestamp (event-time)
        {"name": "ingestion_time", "type": "string"},  # ISO timestamp (arrival/processing time)
        {"name": "customer_id", "type": "string"},
        {"name": "restaurant_id", "type": "string"},
        {"name": "courier_id", "type": ["null", "string"], "default": None},
        {"name": "zone_id", "type": "string"},
        {"name": "estimated_prep_min", "type": ["null", "int"], "default": None},
        {"name": "estimated_delivery_min", "type": ["null", "int"], "default": None},
        {"name": "total_amount_eur", "type": "float"},
        {"name": "promo_applied", "type": "boolean"},
        {"name": "cancel_reason", "type": ["null", "string"], "default": None}
    ]
}

# ============================================================
# Feed B: Courier Status/Location Events (JSON + AVRO schema)
# ============================================================

courier_schema_v1 = {
    "type": "record",
    "name": "CourierStatusEvent",
    "namespace": "food.delivery",
    "doc": "Courier availability + location + status events",
    "fields": [
        {"name": "schema_version", "type": "string", "default": "1.0"},
        {"name": "event_id", "type": "string"},
        {"name": "courier_id", "type": "string"},
        {"name": "event_type", "type": {"type": "enum", "name": "CourierEventType",
                                        "symbols": ["ONLINE", "OFFLINE", "LOCATION", "ASSIGNED", "UNASSIGNED"]}},
        {"name": "event_time", "type": "string"},      # ISO timestamp (event-time)
        {"name": "ingestion_time", "type": "string"},  # ISO timestamp (arrival/processing time)
        {"name": "zone_id", "type": "string"},
        {"name": "lat", "type": "float"},
        {"name": "lon", "type": "float"},
        {"name": "status", "type": {"type": "enum", "name": "CourierStatus",
                                    "symbols": ["IDLE", "EN_ROUTE_TO_RESTAURANT", "WAITING", "EN_ROUTE_TO_CUSTOMER", "OFFLINE"]}},
        {"name": "current_order_id", "type": ["null", "string"], "default": None},
        {"name": "battery_pct", "type": "int"}
    ]
}

# ============================================================
# Helpers: time + realism
# ============================================================

def iso(dt: datetime.datetime) -> str:
    return dt.replace(microsecond=0).isoformat()

def rand_dt(start: datetime.datetime, end: datetime.datetime) -> datetime.datetime:
    """Uniform random datetime."""
    delta = end - start
    seconds = random.randint(0, int(delta.total_seconds()) - 1)
    return start + datetime.timedelta(seconds=seconds)

def is_weekend(dt: datetime.datetime) -> bool:
    return dt.weekday() >= 5

def peak_weighted_time():
    """
    Pick a timestamp with lunch/dinner peaks and weekend uplift.
    """
    dt = rand_dt(START_DATE, END_DATE)
    hour = dt.hour

    weight = 1.0
    if hour in LUNCH_HOURS:
        weight *= 2.3
    if hour in DINNER_HOURS:
        weight *= 2.8
    if is_weekend(dt):
        weight *= WEEKEND_MULTIPLIER

    # Rejection sampling: accept with prob proportional to weight (cap at ~3.5)
    cap = 3.5
    if random.random() < min(1.0, weight / cap):
        return dt
    return peak_weighted_time()

def pick_zone() -> str:
    return random.choices(ZONES, weights=ZONE_WEIGHTS, k=1)[0]

def zone_center_latlon(zone_id: str):
    """
    Fake zone centroids (roughly city-scale). Adjust if you want.
    """
    centers = {
        "Z1_Center": (40.4168, -3.7038),
        "Z2_North":  (40.4780, -3.6880),
        "Z3_South":  (40.3800, -3.7000),
        "Z4_East":   (40.4200, -3.6000),
        "Z5_West":   (40.4200, -3.8000),
    }
    return centers[zone_id]

def jitter_latlon(lat, lon, meters=800):
    """
    Very rough jitter (not geodesic-accurate, good enough for synthetic).
    """
    # ~111km per degree lat; lon scale depends on lat, but we keep simple
    jitter_deg = meters / 111_000.0
    return (
        lat + random.uniform(-jitter_deg, jitter_deg),
        lon + random.uniform(-jitter_deg, jitter_deg),
    )

def maybe_late(event_time: datetime.datetime) -> datetime.datetime:
    """
    Ingestion time is usually >= event_time; sometimes we simulate late arrivals:
    - event_time may be earlier than other events but ingestion arrives later
    """
    if random.random() < LATE_EVENT_PROB:
        # Late arrival: ingestion comes 2 to 25 minutes after event time
        return event_time + datetime.timedelta(minutes=random.randint(2, 25))
    # Normal: small processing delay
    return event_time + datetime.timedelta(seconds=random.randint(1, 45))

def promo_or_surge_flags(dt: datetime.datetime):
    """
    Promo/surge periods depend on time-of-day and randomness.
    """
    surge = (random.random() < SURGE_PROB) and (dt.hour in (DINNER_HOURS + LUNCH_HOURS))
    promo = (random.random() < PROMO_PROB)
    return promo, surge

def cancellation_probability(promo: bool, surge: bool) -> float:
    p = BASE_CANCEL_PROB
    if surge:
        p += SURGE_CANCEL_BONUS
    if promo:
        p += PROMO_CANCEL_BONUS
    return max(0.0, min(0.40, p))

# ============================================================
# Entity pools
# ============================================================

restaurant_ids = [f"R{str(i).zfill(4)}" for i in range(1, NUM_RESTAURANTS + 1)]
courier_ids    = [f"C{str(i).zfill(5)}" for i in range(1, NUM_COURIERS + 1)]
customer_ids   = [f"U{str(i).zfill(5)}" for i in range(1, NUM_CUSTOMERS + 1)]

# Track some courier state to create plausible sequences
courier_state = {
    cid: {"online": False, "zone": pick_zone(), "lat": 0.0, "lon": 0.0, "status": "OFFLINE", "order": None}
    for cid in courier_ids
}
for cid in courier_ids:
    z = courier_state[cid]["zone"]
    lat, lon = zone_center_latlon(z)
    lat, lon = jitter_latlon(lat, lon)
    courier_state[cid].update({"lat": lat, "lon": lon})

# Track orders for lifecycle generation
order_lifecycle_map = {}  # order_id -> dict about chosen restaurant, courier, etc.

# ============================================================
# Feed A: Generate Order Lifecycle Events
# ============================================================

def gen_order_id(n: int) -> str:
    return f"O{str(n).zfill(8)}"

def gen_event_id(prefix: str) -> str:
    return f"{prefix}_{random.randint(1, 10**12):012d}"

def generate_order_lifecycle_events(num_orders: int):
    """
    Generate lifecycle event sequences per order, then shuffle to simulate stream.
    """
    events = []

    for i in range(1, num_orders + 1):
        order_id = gen_order_id(i)
        zone = pick_zone()
        cust = random.choice(customer_ids)
        rest = random.choice(restaurant_ids)

        t0 = peak_weighted_time()
        promo, surge = promo_or_surge_flags(t0)

        # Base estimates (in minutes)
        est_prep = random.randint(8, 25) + (10 if surge else 0)
        est_delivery = random.randint(12, 40) + (8 if surge else 0)

        # Choose courier later (nullable before assignment)
        courier = random.choice(courier_ids)

        # Cancellation decision
        cancel_p = cancellation_probability(promo, surge)
        will_cancel = random.random() < cancel_p

        # Some orders miss steps (edge case)
        missing_step = random.random() < MISSING_STEP_PROB

        # Some orders have impossible durations (edge case)
        impossible = random.random() < IMPOSSIBLE_DURATION_PROB

        order_lifecycle_map[order_id] = {
            "zone": zone, "customer": cust, "restaurant": rest, "courier": courier,
            "promo": promo, "surge": surge
        }

        def add(event_type, event_time, courier_id=None, cancel_reason=None):
            ingestion = maybe_late(event_time)
            events.append({
                "schema_version": "1.0",
                "event_id": gen_event_id("ORD"),
                "order_id": order_id,
                "event_type": event_type,
                "event_time": iso(event_time),
                "ingestion_time": iso(ingestion),
                "customer_id": cust,
                "restaurant_id": rest,
                "courier_id": courier_id,
                "zone_id": zone,
                "estimated_prep_min": est_prep if event_type in ("CREATED", "ACCEPTED") else None,
                "estimated_delivery_min": est_delivery if event_type in ("CREATED", "ACCEPTED") else None,
                "total_amount_eur": round(random.uniform(8, 55), 2),
                "promo_applied": promo,
                "cancel_reason": cancel_reason
            })

        # Lifecycle timings
        t_created = t0
        t_accepted = t_created + datetime.timedelta(minutes=random.randint(1, 6))
        t_prep = t_accepted + datetime.timedelta(minutes=random.randint(0, 4))
        t_ready = t_prep + datetime.timedelta(minutes=est_prep)

        # pickup after ready usually
        pickup_delay = random.randint(2, 12)
        if impossible:
            # Impossible: courier picks up before ready, or delivery extremely fast/slow
            pickup_delay = -random.randint(1, 6)

        t_pickup = t_ready + datetime.timedelta(minutes=pickup_delay)
        travel = random.randint(10, 35)
        if impossible:
            travel = random.choice([-random.randint(1, 8), random.randint(70, 140)])
        t_delivered = t_pickup + datetime.timedelta(minutes=travel)

        # Emit events
        add("CREATED", t_created, courier_id=None)

        # Sometimes duplicate an event (edge case)
        if random.random() < DUPLICATE_EVENT_PROB:
            add("CREATED", t_created, courier_id=None)

        if will_cancel:
            # Cancel could happen before accepted or after accepted
            cancel_time = t_created + datetime.timedelta(minutes=random.randint(1, 18))
            cancel_reason = random.choice(["CUSTOMER_CHANGED_MIND", "RESTAURANT_TOO_BUSY", "COURIER_UNAVAILABLE"])
            # Include maybe ACCEPTED, maybe not
            if random.random() < 0.55:
                add("ACCEPTED", t_accepted, courier_id=courier)
            add("CANCELLED", cancel_time, courier_id=(courier if random.random() < 0.5 else None), cancel_reason=cancel_reason)
            continue

        add("ACCEPTED", t_accepted, courier_id=courier)

        if not missing_step:
            add("PREP_STARTED", t_prep, courier_id=courier)

        add("READY", t_ready, courier_id=courier)

        # Missing step edge case: delivered without pickup
        if missing_step and random.random() < 0.5:
            add("DELIVERED", t_delivered, courier_id=courier)
        else:
            add("PICKED_UP", t_pickup, courier_id=courier)
            add("DELIVERED", t_delivered, courier_id=courier)

    # Shuffle to simulate interleaving orders and some out-of-order arrivals
    random.shuffle(events)
    return events

# We want about NUM_ORDER_EVENTS total events, not orders.
# Roughly each non-cancel order produces ~6-7 events; cancel orders produce ~2-3.
# So choose num_orders accordingly.
approx_events_per_order = 6
num_orders = max(1, NUM_ORDER_EVENTS // approx_events_per_order)
order_events = generate_order_lifecycle_events(num_orders)

# If we overshot, trim; if undershot, generate more (simple approach)
if len(order_events) > NUM_ORDER_EVENTS:
    order_events = order_events[:NUM_ORDER_EVENTS]
elif len(order_events) < NUM_ORDER_EVENTS:
    extra = generate_order_lifecycle_events(max(1, (NUM_ORDER_EVENTS - len(order_events)) // approx_events_per_order))
    order_events.extend(extra[:(NUM_ORDER_EVENTS - len(order_events))])

# ============================================================
# Feed B: Generate Courier Status/Location Events
# ============================================================

def generate_courier_events(num_events: int):
    events = []
    active_orders_by_courier = defaultdict(lambda: None)

    for _ in range(num_events):
        cid = random.choice(courier_ids)
        st = courier_state[cid]

        # choose a time with similar peak bias (couriers active around peaks)
        t = peak_weighted_time()
        promo, surge = promo_or_surge_flags(t)

        # Sometimes force online/offline toggles
        roll = random.random()
        if not st["online"] and roll < 0.20:
            st["online"] = True
            st["status"] = "IDLE"
            event_type = "ONLINE"
        elif st["online"] and roll < 0.08:
            # go offline
            st["online"] = False
            st["status"] = "OFFLINE"
            st["order"] = None
            event_type = "OFFLINE"
        else:
            # other event types
            if st["online"] and st["order"] is None and random.random() < 0.10:
                # assignment to an order
                # pick a random existing order (could also be "future" order_id to simulate mismatch)
                order_id = random.choice(list(order_lifecycle_map.keys()))
                st["order"] = order_id
                active_orders_by_courier[cid] = order_id
                st["status"] = random.choice(["EN_ROUTE_TO_RESTAURANT", "WAITING"])
                event_type = "ASSIGNED"
            elif st["online"] and st["order"] is not None and random.random() < 0.08:
                # unassign
                st["order"] = None
                active_orders_by_courier[cid] = None
                st["status"] = "IDLE"
                event_type = "UNASSIGNED"
            else:
                event_type = "LOCATION"
                if st["online"]:
                    # move within zone (or sometimes jump zones)
                    if random.random() < 0.04:
                        st["zone"] = pick_zone()
                    base_lat, base_lon = zone_center_latlon(st["zone"])
                    st["lat"], st["lon"] = jitter_latlon(base_lat, base_lon, meters=1200)
                    if st["order"] is None:
                        st["status"] = "IDLE"
                    else:
                        st["status"] = random.choice(["EN_ROUTE_TO_RESTAURANT", "WAITING", "EN_ROUTE_TO_CUSTOMER"])
                else:
                    st["status"] = "OFFLINE"

        # courier offline mid-delivery (edge case)
        if st["online"] and st["order"] is not None and random.random() < COURIER_OFFLINE_MID_DELIVERY_PROB:
            # emit an OFFLINE right after (same event_time or close)
            offline_time = t + datetime.timedelta(seconds=random.randint(5, 50))
            ingestion = maybe_late(offline_time)
            events.append({
                "schema_version": "1.0",
                "event_id": gen_event_id("CUR"),
                "courier_id": cid,
                "event_type": "OFFLINE",
                "event_time": iso(offline_time),
                "ingestion_time": iso(ingestion),
                "zone_id": st["zone"],
                "lat": float(st["lat"]),
                "lon": float(st["lon"]),
                "status": "OFFLINE",
                "current_order_id": st["order"],
                "battery_pct": random.randint(1, 100)
            })
            st["online"] = False
            st["status"] = "OFFLINE"
            st["order"] = None

        ingestion = maybe_late(t)

        evt = {
            "schema_version": "1.0",
            "event_id": gen_event_id("CUR"),
            "courier_id": cid,
            "event_type": event_type,
            "event_time": iso(t),
            "ingestion_time": iso(ingestion),
            "zone_id": st["zone"],
            "lat": float(st["lat"]),
            "lon": float(st["lon"]),
            "status": st["status"],
            "current_order_id": st["order"],
            "battery_pct": random.randint(1, 100)
        }
        events.append(evt)

        # duplicates (edge case)
        if random.random() < DUPLICATE_EVENT_PROB:
            events.append(dict(evt))

    random.shuffle(events)
    return events

courier_events = generate_courier_events(NUM_COURIER_EVENTS)
import json

# ------------------------------------------------------------
# JSON serialization (exact pattern you showed)
# ------------------------------------------------------------

# Feed A: Orders (JSON)
with open('order_lifecycle_events.json', 'w') as out:
    json.dump(order_events, out,indent=2)

with open('order_lifecycle_events.json', 'rb') as f:
    json_bytes = f.read()

len(json_bytes), len(json_bytes) / (2**10)  # bytes, KB


# Feed B: Couriers (JSON)
with open('courier_status_events.json', 'w') as out:
    json.dump(courier_events, out,indent=2)

with open('courier_status_events.json', 'rb') as f:
    json_bytes = f.read()

len(json_bytes), len(json_bytes) / (2**10)  # bytes, KB


# ------------------------------------------------------------
# AVRO serialization (exact pattern you showed)
# ------------------------------------------------------------

# Writing
from fastavro import writer, parse_schema

# Feed A: Orders (AVRO)
# The schema variable is defined earlier: order_schema_v1
parsed_schema = parse_schema(order_schema_v1)

with open('order_lifecycle_events.avro', 'wb') as out:
    writer(out, parsed_schema, order_events)

# The AVRO file contains schema + binary data; preview first bytes
with open('order_lifecycle_events.avro', 'rb') as out:
    schema_plus_record = out.read()

schema_plus_record[:1000]


# Feed B: Couriers (AVRO)
# The schema variable is defined earlier: courier_schema_v1
parsed_schema = parse_schema(courier_schema_v1)

with open('courier_status_events.avro', 'wb') as out:
    writer(out, parsed_schema, courier_events)

with open('courier_status_events.avro', 'rb') as out:
    schema_plus_record = out.read()

schema_plus_record[:1000]

b'Obj\x01\x04\x14avro.codec\x08null\x16avro.schema\xc4\x0e{"type": "record", "doc": "Courier availability + location + status events", "name": "food.delivery.CourierStatusEvent", "fields": [{"default": "1.0", "name": "schema_version", "type": "string"}, {"name": "event_id", "type": "string"}, {"name": "courier_id", "type": "string"}, {"name": "event_type", "type": {"type": "enum", "name": "food.delivery.CourierEventType", "symbols": ["ONLINE", "OFFLINE", "LOCATION", "ASSIGNED", "UNASSIGNED"]}}, {"name": "event_time", "type": "string"}, {"name": "ingestion_time", "type": "string"}, {"name": "zone_id", "type": "string"}, {"name": "lat", "type": "float"}, {"name": "lon", "type": "float"}, {"name": "status", "type": {"type": "enum", "name": "food.delivery.CourierStatus", "symbols": ["IDLE", "EN_ROUTE_TO_RESTAURANT", "WAITING", "EN_ROUTE_TO_CUSTOMER", "OFFLINE"]}}, {"default": null, "name": "current_order_id", "type": ["null", "string"]}, {"name": "battery_pct", "type": "int"}]}\x00\x1f\x17

In [5]:
# ── 1. Save AVRO schemas as standalone .avsc files ───────────────────
with open('order_lifecycle_events.avsc', 'w') as f:
    json.dump(order_schema_v1, f, indent=2)
print("✅ order_lifecycle_events.avsc written")

with open('courier_status_events.avsc', 'w') as f:
    json.dump(courier_schema_v1, f, indent=2)
print("✅ courier_status_events.avsc written")

# ── 2. Sample JSON (first 10 events from each feed) ──────────────────
with open('sample_order_lifecycle_events.json', 'w') as f:
    json.dump(order_events[:10], f, indent=2)
print("✅ sample_order_lifecycle_events.json written")

with open('sample_courier_status_events.json', 'w') as f:
    json.dump(courier_events[:10], f, indent=2)
print("✅ sample_courier_status_events.json written")

# ── 3. Sample AVRO (first 10 events from each feed) ──────────────────
from fastavro import writer, parse_schema

parsed_order_schema = parse_schema(order_schema_v1)
with open('sample_order_lifecycle_events.avro', 'wb') as f:
    writer(f, parsed_order_schema, order_events[:10])
print("✅ sample_order_lifecycle_events.avro written")

parsed_courier_schema = parse_schema(courier_schema_v1)
with open('sample_courier_status_events.avro', 'wb') as f:
    writer(f, parsed_courier_schema, courier_events[:10])
print("✅ sample_courier_status_events.avro written")

print("\n📂 All missing deliverables generated.")

✅ order_lifecycle_events.avsc written
✅ courier_status_events.avsc written
✅ sample_order_lifecycle_events.json written
✅ sample_courier_status_events.json written
✅ sample_order_lifecycle_events.avro written
✅ sample_courier_status_events.avro written

📂 All missing deliverables generated.
